In [ ]:
from pathlib import Path

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "python").is_dir() and (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from the "
        "credit-risk-analytics project folder or one of its subfolders."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
LOCAL_DATA_DIR = PROJECT_ROOT / "local_data" / "processed"
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "local_outputs"

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_DIR)
print("Local processed-data folder:", LOCAL_DATA_DIR)
print("Local output folder:", LOCAL_OUTPUT_DIR)


In [ ]:
import numpy as np
import pandas as pd
import scipy
import sklearn

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\n🎉 CreditWise environment is working!")

In [ ]:
import pandas as pd

# Load prepared training and validation data

X_train = pd.read_csv(LOCAL_DATA_DIR / "X_train.csv")
X_valid = pd.read_csv(LOCAL_DATA_DIR / "X_valid.csv")

y_train = pd.read_csv(LOCAL_DATA_DIR / "y_train.csv").squeeze()
y_valid = pd.read_csv(LOCAL_DATA_DIR / "y_valid.csv").squeeze()

print("Data loaded successfully.")

print("\nX_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

In [ ]:
# Identify numerical and categorical predictors

target_col = "TARGET"
id_col = "SK_ID_CURR"

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print(numeric_features)

print("\nNumber of categorical features:", len(categorical_features))
print(categorical_features)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Logistic regression pipeline created successfully.")

In [ ]:
logistic_model.fit(X_train, y_train)

print("Logistic regression model trained successfully.")

In [ ]:
# Generate predictions on the validation dataset

y_pred = logistic_model.predict(X_valid)
y_pred_proba = logistic_model.predict_proba(X_valid)[:, 1]

print("Predictions generated successfully.")

print("\nFirst 10 predicted probabilities of default:")
print(y_pred_proba[:10])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

accuracy = accuracy_score(y_valid, y_pred)
precision = precision_score(y_valid, y_pred)
recall = recall_score(y_valid, y_pred)
f1 = f1_score(y_valid, y_pred)
roc_auc = roc_auc_score(y_valid, y_pred_proba)
pr_auc = average_precision_score(y_valid, y_pred_proba)

print("Logistic Regression Performance")
print("=" * 45)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_valid, y_pred)

print("Confusion Matrix")
print("=" * 30)
print(cm)

In [ ]:
# ============================================================
# LOGISTIC REGRESSION INTERPRETATION
# Extract transformed feature names and coefficients
# ============================================================

# Get the fitted preprocessing component
fitted_preprocessor = logistic_model.named_steps["preprocessor"]

# Get the fitted logistic regression classifier
fitted_classifier = logistic_model.named_steps["classifier"]

# Extract the transformed feature names
feature_names = fitted_preprocessor.get_feature_names_out()

# Extract coefficients
coefficients = fitted_classifier.coef_[0]

print("Number of transformed features:", len(feature_names))
print("Number of coefficients:", len(coefficients))

print("\nFirst 20 transformed features:")
print(feature_names[:20])

In [ ]:
# Create coefficient dataframe

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

# Add absolute coefficient magnitude
coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()

# Sort by absolute magnitude
coef_df = coef_df.sort_values(
    "Absolute_Coefficient",
    ascending=False
).reset_index(drop=True)

print("Coefficient table created successfully.")

coef_df.head(20)

In [ ]:
import numpy as np

In [ ]:
# Calculate odds ratios

coef_df["Odds_Ratio"] = np.exp(coef_df["Coefficient"])

# Percentage change in odds
coef_df["Odds_Change_%"] = (
    (coef_df["Odds_Ratio"] - 1) * 100
)

# Display strongest effects
coef_df[
    [
        "Feature",
        "Coefficient",
        "Odds_Ratio",
        "Odds_Change_%"
    ]
].head(20)

In [ ]:
# ============================================================
# RISK-INCREASING FEATURES
# ============================================================

risk_increasing = coef_df[
    coef_df["Coefficient"] > 0
].sort_values(
    "Coefficient",
    ascending=False
)

print("Top 15 features associated with INCREASED default odds:")
display(
    risk_increasing[
        [
            "Feature",
            "Coefficient",
            "Odds_Ratio",
            "Odds_Change_%"
        ]
    ].head(15)
)


# ============================================================
# RISK-REDUCING FEATURES
# ============================================================

risk_reducing = coef_df[
    coef_df["Coefficient"] < 0
].sort_values(
    "Coefficient",
    ascending=True
)

print("\nTop 15 features associated with REDUCED default odds:")
display(
    risk_reducing[
        [
            "Feature",
            "Coefficient",
            "Odds_Ratio",
            "Odds_Change_%"
        ]
    ].head(15)
)

In [ ]:
# Identify coefficient rows associated with numerical predictors

numerical_coefs = coef_df[
    coef_df["Feature"].str.startswith("num__")
].copy()

numerical_coefs["Original_Feature"] = (
    numerical_coefs["Feature"]
    .str.replace("num__", "", regex=False)
)

numerical_coefs = numerical_coefs.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

print("Numerical predictors ranked by coefficient magnitude:")

display(
    numerical_coefs[
        [
            "Original_Feature",
            "Coefficient",
            "Odds_Ratio",
            "Odds_Change_%"
        ]
    ]
)

In [ ]:
# Extract categorical coefficients

categorical_coefs = coef_df[
    coef_df["Feature"].str.startswith("cat__")
].copy()

categorical_coefs["Original_Feature"] = (
    categorical_coefs["Feature"]
    .str.replace("cat__", "", regex=False)
)

categorical_coefs = categorical_coefs.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

print("Strongest categorical effects:")

display(
    categorical_coefs[
        [
            "Original_Feature",
            "Coefficient",
            "Odds_Ratio",
            "Odds_Change_%"
        ]
    ].head(30)
)

In [ ]:
# ============================================================
# SAVE BASELINE MODEL PERFORMANCE
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

baseline_results = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [accuracy_score(y_valid, y_pred)],
    "Precision": [precision_score(y_valid, y_pred)],
    "Recall": [recall_score(y_valid, y_pred)],
    "F1": [f1_score(y_valid, y_pred)],
    "ROC_AUC": [roc_auc_score(y_valid, y_pred_proba)],
    "PR_AUC": [average_precision_score(y_valid, y_pred_proba)]
})

display(baseline_results)

In [ ]:
# Create prediction dataframe

prediction_analysis = pd.DataFrame({
    "Actual": y_valid.values,
    "Predicted_Probability": y_pred_proba,
    "Predicted_Class": y_pred
})

print("Prediction analysis dataset created.")

display(prediction_analysis.head(10))

print("\nProbability summary:")
display(
    prediction_analysis["Predicted_Probability"].describe()
)

In [ ]:
# ============================================================
# PREDICTED RISK BANDS
# ============================================================

prediction_analysis["Risk_Band"] = pd.cut(
    prediction_analysis["Predicted_Probability"],
    bins=[0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00],
    labels=[
        "0-10%",
        "10-20%",
        "20-30%",
        "30-40%",
        "40-50%",
        "50-60%",
        "60-70%",
        "70-80%",
        "80-90%",
        "90-100%"
    ],
    include_lowest=True
)

risk_band_summary = (
    prediction_analysis
    .groupby("Risk_Band", observed=False)
    .agg(
        Applicant_Count=("Actual", "count"),
        Actual_Defaults=("Actual", "sum"),
        Actual_Default_Rate=("Actual", "mean"),
        Average_Predicted_Risk=("Predicted_Probability", "mean")
    )
    .reset_index()
)

risk_band_summary["Actual_Default_Rate_%"] = (
    risk_band_summary["Actual_Default_Rate"] * 100
)

display(risk_band_summary)

In [ ]:
from sklearn.calibration import calibration_curve

# Calculate calibration values

prob_true, prob_pred = calibration_curve(
    y_valid,
    y_pred_proba,
    n_bins=10,
    strategy="quantile"
)

calibration_df = pd.DataFrame({
    "Mean_Predicted_Probability": prob_pred,
    "Actual_Default_Rate": prob_true
})

calibration_df["Difference"] = (
    calibration_df["Actual_Default_Rate"]
    - calibration_df["Mean_Predicted_Probability"]
)

display(calibration_df)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(
    y_valid,
    y_pred_proba
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc_score(y_valid, y_pred_proba):.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Logistic Regression")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, pr_thresholds = precision_recall_curve(
    y_valid,
    y_pred_proba
)

plt.figure(figsize=(8, 6))

plt.plot(
    recall,
    precision,
    label=f"Logistic Regression (PR-AUC = {average_precision_score(y_valid, y_pred_proba):.3f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Logistic Regression")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# THRESHOLD ANALYSIS
# ============================================================

thresholds_to_test = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80
]

threshold_results = []

for threshold in thresholds_to_test:

    threshold_pred = (
        y_pred_proba >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "Accuracy": accuracy_score(
            y_valid,
            threshold_pred
        )
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(threshold_results)

In [ ]:
best_f1_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

print("Threshold with highest F1-score:")
display(
    best_f1_row.to_frame().T
)

In [ ]:
# ============================================================
# EXPORT LOGISTIC REGRESSION RESULTS
# ============================================================

coef_df.to_csv(LOCAL_OUTPUT_DIR / "logistic_regression_coefficients.csv",
    index=False
)

baseline_results.to_csv(LOCAL_OUTPUT_DIR / "logistic_regression_baseline_results.csv",
    index=False
)

risk_band_summary.to_csv(LOCAL_OUTPUT_DIR / "logistic_regression_risk_bands.csv",
    index=False
)

calibration_df.to_csv(LOCAL_OUTPUT_DIR / "logistic_regression_calibration.csv",
    index=False
)

threshold_results.to_csv(LOCAL_OUTPUT_DIR / "logistic_regression_threshold_analysis.csv",
    index=False
)

print("All Logistic Regression analysis outputs exported successfully.")

In [ ]:
# Create Random Forest preprocessing and modelling pipeline

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Numerical preprocessing
numeric_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_rf, numeric_features),
        ("cat", categorical_transformer_rf, categorical_features)
    ]
)

# Random Forest model
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest pipeline created successfully.")

In [ ]:
# Train the Random Forest pipeline

random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

In [ ]:
# Generate Random Forest predictions

rf_pred_proba = random_forest_model.predict_proba(X_valid)[:, 1]

rf_pred = (rf_pred_proba >= 0.50).astype(int)

print("Random Forest predictions generated successfully.")

print("\nFirst 10 predicted probabilities of default:")
print(rf_pred_proba[:10])

In [ ]:
# Evaluate Random Forest performance

rf_accuracy = accuracy_score(y_valid, rf_pred)
rf_precision = precision_score(y_valid, rf_pred, zero_division=0)
rf_recall = recall_score(y_valid, rf_pred, zero_division=0)
rf_f1 = f1_score(y_valid, rf_pred, zero_division=0)
rf_roc_auc = roc_auc_score(y_valid, rf_pred_proba)
rf_pr_auc = average_precision_score(y_valid, rf_pred_proba)

print("Random Forest Performance")
print("=" * 45)

print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-score:  {rf_f1:.4f}")
print(f"ROC-AUC:   {rf_roc_auc:.4f}")
print(f"PR-AUC:    {rf_pr_auc:.4f}")

In [ ]:
# Random Forest Threshold Analysis

rf_thresholds = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80
]

rf_threshold_results = []

for threshold in rf_thresholds:

    rf_threshold_pred = (
        rf_pred_proba >= threshold
    ).astype(int)

    rf_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_valid,
            rf_threshold_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_valid,
            rf_threshold_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_valid,
            rf_threshold_pred,
            zero_division=0
        ),
        "Accuracy": accuracy_score(
            y_valid,
            rf_threshold_pred
        )
    })

rf_threshold_results = pd.DataFrame(
    rf_threshold_results
)

display(rf_threshold_results)

In [ ]:
# Clean Model Comparison Table

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        0.6836,
        rf_accuracy
    ],
    "Precision": [
        0.1576,
        rf_precision
    ],
    "Recall": [
        0.6719,
        rf_recall
    ],
    "F1": [
        0.2553,
        rf_f1
    ],
    "ROC-AUC": [
        0.7458,
        rf_roc_auc
    ],
    "PR-AUC": [
        0.2289,
        rf_pr_auc
    ]
})

display(model_comparison)

In [ ]:
# ============================================================
# MODEL 3: GRADIENT BOOSTING
# ============================================================

from sklearn.ensemble import GradientBoostingClassifier

# Create Gradient Boosting model
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

print("Gradient Boosting model created successfully.")

In [ ]:
# Train Gradient Boosting using the preprocessing pipeline

gradient_boosting_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gradient_boosting_model.fit(
    X_train,
    y_train
)

print("Gradient Boosting model trained successfully.")

In [ ]:
# Generate Gradient Boosting predictions

gb_pred = gradient_boosting_model.predict(X_valid)
gb_pred_proba = gradient_boosting_model.predict_proba(X_valid)[:, 1]

print("Gradient Boosting predictions generated successfully.")

print("\nFirst 10 predicted probabilities of default:")
print(gb_pred_proba[:10])

In [ ]:
# Evaluate Gradient Boosting performance

gb_accuracy = accuracy_score(y_valid, gb_pred)
gb_precision = precision_score(y_valid, gb_pred, zero_division=0)
gb_recall = recall_score(y_valid, gb_pred, zero_division=0)
gb_f1 = f1_score(y_valid, gb_pred, zero_division=0)
gb_roc_auc = roc_auc_score(y_valid, gb_pred_proba)
gb_pr_auc = average_precision_score(y_valid, gb_pred_proba)

print("Gradient Boosting Performance")
print("=============================================")
print(f"Accuracy:  {gb_accuracy:.4f}")
print(f"Precision: {gb_precision:.4f}")
print(f"Recall:    {gb_recall:.4f}")
print(f"F1-score:  {gb_f1:.4f}")
print(f"ROC-AUC:   {gb_roc_auc:.4f}")
print(f"PR-AUC:    {gb_pr_auc:.4f}")

In [ ]:
# Confusion Matrix for Gradient Boosting

gb_cm = confusion_matrix(y_valid, gb_pred)

print("Gradient Boosting Confusion Matrix")
print("=============================================")
print(gb_cm)

In [ ]:
# Gradient Boosting Threshold Analysis

gb_threshold_results = []

for threshold in thresholds_to_test:

    threshold_pred = (
        gb_pred_proba >= threshold
    ).astype(int)

    gb_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "Accuracy": accuracy_score(
            y_valid,
            threshold_pred
        )
    })

gb_threshold_results = pd.DataFrame(
    gb_threshold_results
)

display(gb_threshold_results)

In [ ]:
# Identify the threshold with the highest F1-score

best_gb_threshold = gb_threshold_results.loc[
    gb_threshold_results["F1"].idxmax()
]

print("Best Gradient Boosting threshold based on F1-score:")
print(best_gb_threshold)

In [ ]:
# Display Gradient Boosting thresholds ranked by F1-score

gb_threshold_ranked = gb_threshold_results.sort_values(
    by="F1",
    ascending=False
)

display(gb_threshold_ranked)

In [ ]:
# ============================================================
# FIND BEST F1 THRESHOLD FOR EACH MODEL
# ============================================================

# Logistic Regression
best_logistic = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

# Random Forest
best_rf = threshold_results.loc[
    threshold_results["F1"].idxmax()
] if "rf_threshold_results" not in globals() else rf_threshold_results.loc[
    rf_threshold_results["F1"].idxmax()
]

# Gradient Boosting
best_gb = gb_threshold_results.loc[
    gb_threshold_results["F1"].idxmax()
]

print("Best Logistic Regression threshold:")
print(best_logistic)

print("\nBest Random Forest threshold:")
print(best_rf)

print("\nBest Gradient Boosting threshold:")
print(best_gb)

In [ ]:
# ============================================================
# RANDOM FOREST THRESHOLD ANALYSIS - CLEAN VERSION
# ============================================================

rf_threshold_results = []

for threshold in thresholds_to_test:

    threshold_pred = (
        rf_pred_proba >= threshold
    ).astype(int)

    rf_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_valid,
            threshold_pred,
            zero_division=0
        ),
        "Accuracy": accuracy_score(
            y_valid,
            threshold_pred
        )
    })

rf_threshold_results = pd.DataFrame(
    rf_threshold_results
)

display(rf_threshold_results)

In [ ]:
# ============================================================
# BEST F1 THRESHOLD FOR ALL THREE MODELS
# ============================================================

best_logistic = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

best_rf = rf_threshold_results.loc[
    rf_threshold_results["F1"].idxmax()
]

best_gb = gb_threshold_results.loc[
    gb_threshold_results["F1"].idxmax()
]

print("BEST LOGISTIC REGRESSION")
print(best_logistic)

print("\nBEST RANDOM FOREST")
print(best_rf)

print("\nBEST GRADIENT BOOSTING")
print(best_gb)

In [ ]:
# ============================================================
# MODEL COMPARISON AT BEST F1 THRESHOLD
# ============================================================

optimal_threshold_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Optimal Threshold": [
        best_logistic["Threshold"],
        best_rf["Threshold"],
        best_gb["Threshold"]
    ],
    "Precision": [
        best_logistic["Precision"],
        best_rf["Precision"],
        best_gb["Precision"]
    ],
    "Recall": [
        best_logistic["Recall"],
        best_rf["Recall"],
        best_gb["Recall"]
    ],
    "F1": [
        best_logistic["F1"],
        best_rf["F1"],
        best_gb["F1"]
    ],
    "Accuracy": [
        best_logistic["Accuracy"],
        best_rf["Accuracy"],
        best_gb["Accuracy"]
    ]
})

display(optimal_threshold_comparison)

In [ ]:
# ============================================================
# ROC CURVE COMPARISON
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Calculate ROC curves
logistic_fpr, logistic_tpr, _ = roc_curve(
    y_valid,
    y_pred_proba
)

rf_fpr, rf_tpr, _ = roc_curve(
    y_valid,
    rf_pred_proba
)

gb_fpr, gb_tpr, _ = roc_curve(
    y_valid,
    gb_pred_proba
)

# Calculate AUC values
logistic_auc = roc_auc_score(
    y_valid,
    y_pred_proba
)

rf_auc = roc_auc_score(
    y_valid,
    rf_pred_proba
)

gb_auc = roc_auc_score(
    y_valid,
    gb_pred_proba
)

# Plot
plt.figure(figsize=(10, 7))

plt.plot(
    logistic_fpr,
    logistic_tpr,
    label=f"Logistic Regression (AUC = {logistic_auc:.4f})"
)

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {rf_auc:.4f})"
)

plt.plot(
    gb_fpr,
    gb_tpr,
    label=f"Gradient Boosting (AUC = {gb_auc:.4f})"
)

# Random classifier reference line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Print ROC-AUC comparison

print("ROC-AUC Comparison")
print("==============================")
print(f"Logistic Regression: {logistic_auc:.4f}")
print(f"Random Forest:       {rf_auc:.4f}")
print(f"Gradient Boosting:   {gb_auc:.4f}")

In [ ]:
# Check the structure of the Random Forest model

print(random_forest_model)

In [ ]:
# Check the available steps in the Random Forest pipeline

print(random_forest_model.named_steps)

In [ ]:
# ============================================================
# RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

# Get feature names after preprocessing
feature_names = random_forest_model.named_steps[
    "preprocessor"
].get_feature_names_out()

# Get feature importance from Random Forest classifier
rf_importance = random_forest_model.named_steps[
    "classifier"
].feature_importances_

# Create feature importance DataFrame
rf_feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_importance
})

# Sort from most to least important
rf_feature_importance = rf_feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Display top 20 features
display(
    rf_feature_importance.head(20)
)

In [ ]:
# ============================================================
# RANDOM FOREST TOP 20 FEATURE IMPORTANCE
# ============================================================

top_rf_features = (
    rf_feature_importance
    .head(20)
    .sort_values(by="Importance")
)

plt.figure(figsize=(10, 8))

plt.barh(
    top_rf_features["Feature"],
    top_rf_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 20 Feature Importances - Random Forest")

plt.tight_layout()
plt.show()

In [ ]:
# Check the Gradient Boosting pipeline structure

print(gradient_boosting_model)
print(gradient_boosting_model.named_steps.keys())

In [ ]:
# ============================================================
# GRADIENT BOOSTING FEATURE IMPORTANCE
# ============================================================

# Get feature names after preprocessing
feature_names = gradient_boosting_model.named_steps[
    "preprocessor"
].get_feature_names_out()

# Get feature importance from Gradient Boosting model
gb_importance = gradient_boosting_model.named_steps[
    "model"
].feature_importances_

# Create feature importance DataFrame
gb_feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": gb_importance
})

# Sort from most to least important
gb_feature_importance = gb_feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Display top 20 features
display(
    gb_feature_importance.head(20)
)

In [ ]:
# ============================================================
# GRADIENT BOOSTING TOP 20 FEATURE IMPORTANCE
# ============================================================

top_gb_features = (
    gb_feature_importance
    .head(20)
    .sort_values(by="Importance")
)

plt.figure(figsize=(10, 8))

plt.barh(
    top_gb_features["Feature"],
    top_gb_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 20 Feature Importances - Gradient Boosting")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MODEL COMPARISON - ROC AND PRECISION-RECALL CURVES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

# ------------------------------------------------------------
# Generate predicted probabilities directly from each model
# ------------------------------------------------------------

logistic_proba = logistic_model.predict_proba(X_valid)[:, 1]

random_forest_proba = random_forest_model.predict_proba(X_valid)[:, 1]

gradient_boosting_proba = gradient_boosting_model.predict_proba(X_valid)[:, 1]


# ============================================================
# 1. ROC CURVES
# ============================================================

logistic_fpr, logistic_tpr, _ = roc_curve(
    y_valid,
    logistic_proba
)

rf_fpr, rf_tpr, _ = roc_curve(
    y_valid,
    random_forest_proba
)

gb_fpr, gb_tpr, _ = roc_curve(
    y_valid,
    gradient_boosting_proba
)


logistic_auc = auc(
    logistic_fpr,
    logistic_tpr
)

rf_auc = auc(
    rf_fpr,
    rf_tpr
)

gb_auc = auc(
    gb_fpr,
    gb_tpr
)


plt.figure(figsize=(9, 7))

plt.plot(
    logistic_fpr,
    logistic_tpr,
    label=f"Logistic Regression (AUC = {logistic_auc:.4f})"
)

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {rf_auc:.4f})"
)

plt.plot(
    gb_fpr,
    gb_tpr,
    label=f"Gradient Boosting (AUC = {gb_auc:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")

plt.legend()

plt.grid(alpha=0.3)

plt.show()


print("ROC-AUC Comparison")
print("==============================")
print(f"Logistic Regression : {logistic_auc:.4f}")
print(f"Random Forest       : {rf_auc:.4f}")
print(f"Gradient Boosting   : {gb_auc:.4f}")


# ============================================================
# 2. PRECISION-RECALL CURVES
# ============================================================

logistic_precision, logistic_recall, _ = precision_recall_curve(
    y_valid,
    logistic_proba
)

rf_precision, rf_recall, _ = precision_recall_curve(
    y_valid,
    random_forest_proba
)

gb_precision, gb_recall, _ = precision_recall_curve(
    y_valid,
    gradient_boosting_proba
)


logistic_pr_auc = average_precision_score(
    y_valid,
    logistic_proba
)

rf_pr_auc = average_precision_score(
    y_valid,
    random_forest_proba
)

gb_pr_auc = average_precision_score(
    y_valid,
    gradient_boosting_proba
)


plt.figure(figsize=(9, 7))

plt.plot(
    logistic_recall,
    logistic_precision,
    label=f"Logistic Regression (PR-AUC = {logistic_pr_auc:.4f})"
)

plt.plot(
    rf_recall,
    rf_precision,
    label=f"Random Forest (PR-AUC = {rf_pr_auc:.4f})"
)

plt.plot(
    gb_recall,
    gb_precision,
    label=f"Gradient Boosting (PR-AUC = {gb_pr_auc:.4f})"
)

# Baseline = proportion of positive observations
baseline = y_valid.mean()

plt.axhline(
    y=baseline,
    linestyle="--",
    label=f"Baseline Default Rate = {baseline:.4f}"
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title("Precision-Recall Curve Comparison")

plt.legend()

plt.grid(alpha=0.3)

plt.show()


print("\nPrecision-Recall AUC Comparison")
print("==============================")
print(f"Logistic Regression : {logistic_pr_auc:.4f}")
print(f"Random Forest       : {rf_pr_auc:.4f}")
print(f"Gradient Boosting   : {gb_pr_auc:.4f}")


# ============================================================
# 3. CREATE MODEL COMPARISON TABLE
# ============================================================

model_comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],

    "ROC_AUC": [
        logistic_auc,
        rf_auc,
        gb_auc
    ],

    "PR_AUC": [
        logistic_pr_auc,
        rf_pr_auc,
        gb_pr_auc
    ]
})


model_comparison = model_comparison.sort_values(
    by="PR_AUC",
    ascending=False
).reset_index(drop=True)


display(model_comparison)


print("\nModel comparison completed successfully.")

In [ ]:
# ============================================================
# MODEL CALIBRATION ANALYSIS
# ============================================================

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# ------------------------------------------------------------
# Calculate Brier Scores
# ------------------------------------------------------------

logistic_brier = brier_score_loss(
    y_valid,
    logistic_proba
)

rf_brier = brier_score_loss(
    y_valid,
    random_forest_proba
)

gb_brier = brier_score_loss(
    y_valid,
    gradient_boosting_proba
)


print("Brier Score Comparison")
print("==============================")
print(f"Logistic Regression : {logistic_brier:.4f}")
print(f"Random Forest       : {rf_brier:.4f}")
print(f"Gradient Boosting   : {gb_brier:.4f}")


# ============================================================
# CALIBRATION CURVES
# ============================================================

logistic_fraction_pos, logistic_mean_pred = calibration_curve(
    y_valid,
    logistic_proba,
    n_bins=10,
    strategy="quantile"
)

rf_fraction_pos, rf_mean_pred = calibration_curve(
    y_valid,
    random_forest_proba,
    n_bins=10,
    strategy="quantile"
)

gb_fraction_pos, gb_mean_pred = calibration_curve(
    y_valid,
    gradient_boosting_proba,
    n_bins=10,
    strategy="quantile"
)


# ============================================================
# PLOT CALIBRATION CURVES
# ============================================================

plt.figure(figsize=(9, 7))

plt.plot(
    logistic_mean_pred,
    logistic_fraction_pos,
    marker="o",
    label="Logistic Regression"
)

plt.plot(
    rf_mean_pred,
    rf_fraction_pos,
    marker="o",
    label="Random Forest"
)

plt.plot(
    gb_mean_pred,
    gb_fraction_pos,
    marker="o",
    label="Gradient Boosting"
)

# Perfect calibration line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Default Rate")

plt.title("Model Calibration Curves")

plt.legend()

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# CALIBRATION TABLES
# ============================================================
import pandas as pd
def create_calibration_table(y_true, probabilities, model_name, n_bins=10):

    calibration_data = pd.DataFrame({
        "Actual": y_true.values,
        "Predicted_Probability": probabilities
    })

    # Divide predictions into 10 equal-sized groups
    calibration_data["Risk_Band"] = pd.qcut(
        calibration_data["Predicted_Probability"],
        q=n_bins,
        duplicates="drop"
    )

    calibration_table = (
        calibration_data
        .groupby("Risk_Band", observed=True)
        .agg(
            Applicant_Count=("Actual", "count"),
            Mean_Predicted_Risk=("Predicted_Probability", "mean"),
            Observed_Default_Rate=("Actual", "mean")
        )
        .reset_index()
    )

    calibration_table["Model"] = model_name

    calibration_table["Absolute_Difference"] = (
        calibration_table["Mean_Predicted_Risk"]
        - calibration_table["Observed_Default_Rate"]
    ).abs()

    return calibration_table


# ------------------------------------------------------------
# Create calibration tables for all three models
# ------------------------------------------------------------

logistic_calibration = create_calibration_table(
    y_valid,
    logistic_proba,
    "Logistic Regression"
)

rf_calibration = create_calibration_table(
    y_valid,
    random_forest_proba,
    "Random Forest"
)

gb_calibration = create_calibration_table(
    y_valid,
    gradient_boosting_proba,
    "Gradient Boosting"
)


# ------------------------------------------------------------
# Display calibration tables
# ------------------------------------------------------------

print("LOGISTIC REGRESSION CALIBRATION")
print("=" * 70)

display(
    logistic_calibration[
        [
            "Risk_Band",
            "Applicant_Count",
            "Mean_Predicted_Risk",
            "Observed_Default_Rate",
            "Absolute_Difference"
        ]
    ]
)


print("\nRANDOM FOREST CALIBRATION")
print("=" * 70)

display(
    rf_calibration[
        [
            "Risk_Band",
            "Applicant_Count",
            "Mean_Predicted_Risk",
            "Observed_Default_Rate",
            "Absolute_Difference"
        ]
    ]
)


print("\nGRADIENT BOOSTING CALIBRATION")
print("=" * 70)

display(
    gb_calibration[
        [
            "Risk_Band",
            "Applicant_Count",
            "Mean_Predicted_Risk",
            "Observed_Default_Rate",
            "Absolute_Difference"
        ]
    ]
)


print("\nCalibration tables generated successfully.")

In [ ]:
# ============================================================
# CALIBRATION PLOT - MODEL COMPARISON
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 7))

# Logistic Regression
plt.plot(
    logistic_calibration["Mean_Predicted_Risk"],
    logistic_calibration["Observed_Default_Rate"],
    marker="o",
    label="Logistic Regression"
)

# Random Forest
plt.plot(
    rf_calibration["Mean_Predicted_Risk"],
    rf_calibration["Observed_Default_Rate"],
    marker="o",
    label="Random Forest"
)

# Gradient Boosting
plt.plot(
    gb_calibration["Mean_Predicted_Risk"],
    gb_calibration["Observed_Default_Rate"],
    marker="o",
    label="Gradient Boosting"
)

# Perfect calibration line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)

plt.xlabel("Mean Predicted Default Probability")
plt.ylabel("Observed Default Rate")

plt.title("Calibration Comparison of Credit Default Models")

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# IDENTIFY BEST MODEL
# ============================================================

# Use the final model comparison table
final_model_comparison = model_comparison.copy()

# Find the best model for each metric
best_pr_auc_model = final_model_comparison.loc[
    final_model_comparison["PR_AUC"].idxmax(),
    "Model"
]

best_roc_auc_model = final_model_comparison.loc[
    final_model_comparison["ROC_AUC"].idxmax(),
    "Model"
]

# Check whether Brier Score is available
if "Brier_Score" in final_model_comparison.columns:
    
    best_brier_model = final_model_comparison.loc[
        final_model_comparison["Brier_Score"].idxmin(),
        "Model"
    ]
    
else:
    best_brier_model = "Brier Score not available in comparison table"


# ============================================================
# PRINT MODEL SELECTION SUMMARY
# ============================================================

print("Model Selection Summary")
print("=" * 60)

print("Best PR-AUC model:", best_pr_auc_model)
print("Best ROC-AUC model:", best_roc_auc_model)


**Optional dependency note:** The original notebook included a SHAP installation command, but no SHAP analysis follows in this notebook. If SHAP is added later, install it in your Python environment separately.


In [ ]:
# ============================================================
# FINAL MODEL EVALUATION
# GRADIENT BOOSTING
# ============================================================

print("GRADIENT BOOSTING — FINAL MODEL EVALUATION")
print("=" * 70)

# Selected threshold from our threshold analysis
gb_threshold = 0.20

# Generate predictions using the EXISTING Gradient Boosting probabilities
gb_optimal_pred = (
    gb_pred_proba >= gb_threshold
).astype(int)

# ------------------------------------------------------------
# Performance metrics
# ------------------------------------------------------------

gb_final_accuracy = accuracy_score(
    y_valid,
    gb_optimal_pred
)

gb_final_precision = precision_score(
    y_valid,
    gb_optimal_pred,
    zero_division=0
)

gb_final_recall = recall_score(
    y_valid,
    gb_optimal_pred,
    zero_division=0
)

gb_final_f1 = f1_score(
    y_valid,
    gb_optimal_pred,
    zero_division=0
)

gb_final_roc_auc = roc_auc_score(
    y_valid,
    gb_pred_proba
)

gb_final_pr_auc = average_precision_score(
    y_valid,
    gb_pred_proba
)

gb_final_brier = brier_score_loss(
    y_valid,
    gb_pred_proba
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(f"Optimal Threshold : {gb_threshold:.2f}")
print()

print(f"Accuracy          : {gb_final_accuracy:.4f}")
print(f"Precision         : {gb_final_precision:.4f}")
print(f"Recall            : {gb_final_recall:.4f}")
print(f"F1-score          : {gb_final_f1:.4f}")
print(f"ROC-AUC           : {gb_final_roc_auc:.4f}")
print(f"PR-AUC            : {gb_final_pr_auc:.4f}")
print(f"Brier Score       : {gb_final_brier:.4f}")

print()
print("Final Gradient Boosting evaluation completed successfully.")

In [ ]:
# ============================================================
# GRADIENT BOOSTING — FINAL CONFUSION MATRIX
# ============================================================

gb_final_cm = confusion_matrix(
    y_valid,
    gb_optimal_pred
)

print("GRADIENT BOOSTING CONFUSION MATRIX")
print("=" * 70)

print(gb_final_cm)

tn, fp, fn, tp = gb_final_cm.ravel()

print()
print(f"True Negatives  : {tn:,}")
print(f"False Positives : {fp:,}")
print(f"False Negatives : {fn:,}")
print(f"True Positives  : {tp:,}")

In [ ]:
# ============================================================
# GRADIENT BOOSTING — ERROR ANALYSIS
# ============================================================

# Use the existing validation data and predictions
gb_error_analysis = X_valid.copy()

# Add actual outcome
gb_error_analysis["Actual"] = y_valid.values

# Add the EXISTING predicted probabilities
gb_error_analysis["Predicted_Probability"] = gb_pred_proba

# Add the EXISTING optimal-threshold predictions
gb_error_analysis["Predicted"] = gb_optimal_pred

# ------------------------------------------------------------
# Identify prediction type
# ------------------------------------------------------------

gb_error_analysis["Prediction_Type"] = np.select(
    [
        (gb_error_analysis["Actual"] == 0) &
        (gb_error_analysis["Predicted"] == 0),

        (gb_error_analysis["Actual"] == 0) &
        (gb_error_analysis["Predicted"] == 1),

        (gb_error_analysis["Actual"] == 1) &
        (gb_error_analysis["Predicted"] == 0),

        (gb_error_analysis["Actual"] == 1) &
        (gb_error_analysis["Predicted"] == 1)
    ],
    [
        "True Negative",
        "False Positive",
        "False Negative",
        "True Positive"
    ],
    default="Unknown"
)

print("Gradient Boosting error analysis created successfully.")

In [ ]:
# ============================================================
# ERROR TYPE SUMMARY
# ============================================================

error_type_summary = (
    gb_error_analysis["Prediction_Type"]
    .value_counts()
    .to_frame("Count")
)

error_type_summary["Percentage"] = (
    error_type_summary["Count"]
    / len(gb_error_analysis)
    * 100
)

display(error_type_summary)

In [ ]:
# ============================================================
# ERROR TYPE SUMMARY — CLEAR INTERPRETATION
# ============================================================

print("GRADIENT BOOSTING ERROR ANALYSIS")
print("=" * 70)

print(f"True Negatives  : {tn:,}")
print(f"False Positives : {fp:,}")
print(f"False Negatives : {fn:,}")
print(f"True Positives  : {tp:,}")

print()
print(f"Total validation observations: {len(y_valid):,}")

print()
print("False Negative Rate among actual defaulters:")
print(
    f"{fn / (fn + tp) * 100:.2f}%"
)

print()
print("False Positive Rate among actual non-defaulters:")
print(
    f"{fp / (fp + tn) * 100:.2f}%"
)

In [ ]:
# ============================================================
# FALSE NEGATIVE APPLICANTS
# ============================================================

false_negatives = gb_error_analysis[
    gb_error_analysis["Prediction_Type"] == "False Negative"
].copy()

print("FALSE NEGATIVE ANALYSIS")
print("=" * 70)

print(f"Number of false negatives: {len(false_negatives):,}")

print(
    f"Mean predicted probability: "
    f"{false_negatives['Predicted_Probability'].mean():.4f}"
)

print(
    f"Minimum predicted probability: "
    f"{false_negatives['Predicted_Probability'].min():.4f}"
)

print(
    f"Maximum predicted probability: "
    f"{false_negatives['Predicted_Probability'].max():.4f}"
)

In [ ]:
# ============================================================
# FALSE POSITIVE APPLICANTS
# ============================================================

false_positives = gb_error_analysis[
    gb_error_analysis["Prediction_Type"] == "False Positive"
].copy()

print("FALSE POSITIVE ANALYSIS")
print("=" * 70)

print(f"Number of false positives: {len(false_positives):,}")

print(
    f"Mean predicted probability: "
    f"{false_positives['Predicted_Probability'].mean():.4f}"
)

print(
    f"Minimum predicted probability: "
    f"{false_positives['Predicted_Probability'].min():.4f}"
)

print(
    f"Maximum predicted probability: "
    f"{false_positives['Predicted_Probability'].max():.4f}"
)

In [ ]:
# ============================================================
# FALSE POSITIVE APPLICANTS
# ============================================================

false_positives = gb_error_analysis[
    gb_error_analysis["Prediction_Type"] == "False Positive"
].copy()

print("FALSE POSITIVE ANALYSIS")
print("=" * 70)

print(f"Number of false positives: {len(false_positives):,}")

print(
    f"Mean predicted probability: "
    f"{false_positives['Predicted_Probability'].mean():.4f}"
)

print(
    f"Minimum predicted probability: "
    f"{false_positives['Predicted_Probability'].min():.4f}"
)

print(
    f"Maximum predicted probability: "
    f"{false_positives['Predicted_Probability'].max():.4f}"
)

In [ ]:
# ============================================================
# GRADIENT BOOSTING — FEATURE IMPORTANCE
# ============================================================

# Get the preprocessing component from the existing pipeline
gb_preprocessor = gradient_boosting_model.named_steps["preprocessor"]

# Get the Gradient Boosting model from the existing pipeline
gb_classifier = gradient_boosting_model.named_steps["model"]

# Get the feature names created by preprocessing
gb_feature_names = gb_preprocessor.get_feature_names_out()

# Get feature importance values
gb_importance_values = gb_classifier.feature_importances_

# Create feature importance dataframe
gb_feature_importance = pd.DataFrame({
    "Feature": gb_feature_names,
    "Importance": gb_importance_values
})

# Sort from most important to least important
gb_feature_importance = (
    gb_feature_importance
    .sort_values(
        by="Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Gradient Boosting feature importance calculated successfully.")

display(
    gb_feature_importance.head(20)
)

In [ ]:
# ============================================================
# TOP 15 GRADIENT BOOSTING FEATURES
# ============================================================

top_n = 15

top_features = (
    gb_feature_importance
    .head(top_n)
    .sort_values(
        by="Importance",
        ascending=True
    )
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title(
    "Top 15 Features Influencing Gradient Boosting Predictions"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ROC CURVE COMPARISON
# ============================================================

# Logistic Regression
logistic_fpr, logistic_tpr, _ = roc_curve(
    y_valid,
    y_pred_proba
)

# Random Forest
rf_fpr, rf_tpr, _ = roc_curve(
    y_valid,
    rf_pred_proba
)

# Gradient Boosting
gb_fpr, gb_tpr, _ = roc_curve(
    y_valid,
    gb_pred_proba
)

plt.figure(figsize=(9, 7))

plt.plot(
    logistic_fpr,
    logistic_tpr,
    label=f"Logistic Regression (AUC = {roc_auc_score(y_valid, y_pred_proba):.4f})"
)

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {roc_auc_score(y_valid, rf_pred_proba):.4f})"
)

plt.plot(
    gb_fpr,
    gb_tpr,
    label=f"Gradient Boosting (AUC = {roc_auc_score(y_valid, gb_pred_proba):.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PRECISION-RECALL CURVE COMPARISON
# ============================================================

logistic_precision, logistic_recall, _ = precision_recall_curve(
    y_valid,
    y_pred_proba
)

rf_precision, rf_recall, _ = precision_recall_curve(
    y_valid,
    rf_pred_proba
)

gb_precision, gb_recall, _ = precision_recall_curve(
    y_valid,
    gb_pred_proba
)

plt.figure(figsize=(9, 7))

plt.plot(
    logistic_recall,
    logistic_precision,
    label=f"Logistic Regression (PR-AUC = {average_precision_score(y_valid, y_pred_proba):.4f})"
)

plt.plot(
    rf_recall,
    rf_precision,
    label=f"Random Forest (PR-AUC = {average_precision_score(y_valid, rf_pred_proba):.4f})"
)

plt.plot(
    gb_recall,
    gb_precision,
    label=f"Gradient Boosting (PR-AUC = {average_precision_score(y_valid, gb_pred_proba):.4f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRADIENT BOOSTING — THRESHOLD PERFORMANCE
# ============================================================

# Recalculate from the existing Gradient Boosting probabilities
# so the graph is generated directly from the model output.

thresholds = np.arange(
    0.05,
    0.81,
    0.01
)

threshold_precision = []
threshold_recall = []
threshold_f1 = []
threshold_accuracy = []

for threshold in thresholds:

    threshold_prediction = (
        gb_pred_proba >= threshold
    ).astype(int)

    threshold_precision.append(
        precision_score(
            y_valid,
            threshold_prediction,
            zero_division=0
        )
    )

    threshold_recall.append(
        recall_score(
            y_valid,
            threshold_prediction,
            zero_division=0
        )
    )

    threshold_f1.append(
        f1_score(
            y_valid,
            threshold_prediction,
            zero_division=0
        )
    )

    threshold_accuracy.append(
        accuracy_score(
            y_valid,
            threshold_prediction
        )
    )

threshold_plot_df = pd.DataFrame({
    "Threshold": thresholds,
    "Precision": threshold_precision,
    "Recall": threshold_recall,
    "F1": threshold_f1,
    "Accuracy": threshold_accuracy
})

# Identify the best F1 threshold from this finer grid
best_threshold_index = (
    threshold_plot_df["F1"].idxmax()
)

best_gb_threshold = (
    threshold_plot_df.loc[
        best_threshold_index,
        "Threshold"
    ]
)

print(
    f"Best Gradient Boosting threshold "
    f"from fine-grid analysis: {best_gb_threshold:.2f}"
)

display(
    threshold_plot_df.loc[
        best_threshold_index
    ]
)

In [ ]:
# ============================================================
# THRESHOLD VS PRECISION / RECALL / F1
# ============================================================

plt.figure(figsize=(10, 7))

plt.plot(
    threshold_plot_df["Threshold"],
    threshold_plot_df["Precision"],
    label="Precision"
)

plt.plot(
    threshold_plot_df["Threshold"],
    threshold_plot_df["Recall"],
    label="Recall"
)

plt.plot(
    threshold_plot_df["Threshold"],
    threshold_plot_df["F1"],
    label="F1-score"
)

plt.axvline(
    best_gb_threshold,
    linestyle="--",
    label=f"Best F1 Threshold = {best_gb_threshold:.2f}"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Metric Value")
plt.title(
    "Gradient Boosting Performance Across Classification Thresholds"
)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CALIBRATION CURVE COMPARISON
# ============================================================

from sklearn.calibration import calibration_curve

# Calculate calibration values
logistic_fraction, logistic_mean = calibration_curve(
    y_valid,
    y_pred_proba,
    n_bins=10,
    strategy="quantile"
)

rf_fraction, rf_mean = calibration_curve(
    y_valid,
    rf_pred_proba,
    n_bins=10,
    strategy="quantile"
)

gb_fraction, gb_mean = calibration_curve(
    y_valid,
    gb_pred_proba,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(9, 7))

plt.plot(
    logistic_mean,
    logistic_fraction,
    marker="o",
    label="Logistic Regression"
)

plt.plot(
    rf_mean,
    rf_fraction,
    marker="o",
    label="Random Forest"
)

plt.plot(
    gb_mean,
    gb_fraction,
    marker="o",
    label="Gradient Boosting"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect Calibration"
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Default Rate")

plt.title(
    "Calibration Curve Comparison"
)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FINAL MODEL COMPARISON — DYNAMIC
# ============================================================

# ------------------------------------------------------------
# 1. Calculate metrics directly from existing predictions
# ------------------------------------------------------------

# Logistic Regression
logistic_accuracy = accuracy_score(
    y_valid,
    (y_pred_proba >= 0.65).astype(int)
)

logistic_precision = precision_score(
    y_valid,
    (y_pred_proba >= 0.65).astype(int),
    zero_division=0
)

logistic_recall = recall_score(
    y_valid,
    (y_pred_proba >= 0.65).astype(int),
    zero_division=0
)

logistic_f1 = f1_score(
    y_valid,
    (y_pred_proba >= 0.65).astype(int),
    zero_division=0
)

logistic_roc_auc = roc_auc_score(
    y_valid,
    y_pred_proba
)

logistic_pr_auc = average_precision_score(
    y_valid,
    y_pred_proba
)


# Random Forest
rf_accuracy = accuracy_score(
    y_valid,
    (rf_pred_proba >= 0.55).astype(int)
)

rf_precision = precision_score(
    y_valid,
    (rf_pred_proba >= 0.55).astype(int),
    zero_division=0
)

rf_recall = recall_score(
    y_valid,
    (rf_pred_proba >= 0.55).astype(int),
    zero_division=0
)

rf_f1 = f1_score(
    y_valid,
    (rf_pred_proba >= 0.55).astype(int),
    zero_division=0
)

rf_roc_auc = roc_auc_score(
    y_valid,
    rf_pred_proba
)

rf_pr_auc = average_precision_score(
    y_valid,
    rf_pred_proba
)


# Gradient Boosting
gb_accuracy = accuracy_score(
    y_valid,
    gb_optimal_pred
)

gb_precision = precision_score(
    y_valid,
    gb_optimal_pred,
    zero_division=0
)

gb_recall = recall_score(
    y_valid,
    gb_optimal_pred,
    zero_division=0
)

gb_f1 = f1_score(
    y_valid,
    gb_optimal_pred,
    zero_division=0
)

gb_roc_auc = roc_auc_score(
    y_valid,
    gb_pred_proba
)

gb_pr_auc = average_precision_score(
    y_valid,
    gb_pred_proba
)


# ------------------------------------------------------------
# 2. Create final comparison table
# ------------------------------------------------------------

final_model_comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],

    "Accuracy": [
        logistic_accuracy,
        rf_accuracy,
        gb_accuracy
    ],

    "Precision": [
        logistic_precision,
        rf_precision,
        gb_precision
    ],

    "Recall": [
        logistic_recall,
        rf_recall,
        gb_recall
    ],

    "F1": [
        logistic_f1,
        rf_f1,
        gb_f1
    ],

    "ROC_AUC": [
        logistic_roc_auc,
        rf_roc_auc,
        gb_roc_auc
    ],

    "PR_AUC": [
        logistic_pr_auc,
        rf_pr_auc,
        gb_pr_auc
    ],

    "Brier_Score": [
        logistic_brier,
        rf_brier,
        gb_brier
    ],

    "Optimal_Threshold": [
        0.65,
        0.55,
        gb_threshold
    ]
})

# ------------------------------------------------------------
# 3. Round for presentation
# ------------------------------------------------------------

final_model_comparison_display = (
    final_model_comparison
    .copy()
)

numeric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Optimal_Threshold"
]

final_model_comparison_display[numeric_columns] = (
    final_model_comparison_display[numeric_columns]
    .round(4)
)

display(final_model_comparison_display)

In [ ]:
# ============================================================
# FINAL MODEL RANKING
# ============================================================

model_ranking = final_model_comparison.copy()

# Rank models according to PR-AUC
model_ranking["PR_AUC_Rank"] = (
    model_ranking["PR_AUC"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

# Rank according to ROC-AUC
model_ranking["ROC_AUC_Rank"] = (
    model_ranking["ROC_AUC"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

model_ranking = model_ranking.sort_values(
    by="PR_AUC",
    ascending=False
)

print("FINAL MODEL RANKING")
print("=" * 70)

display(
    model_ranking[
        [
            "Model",
            "PR_AUC",
            "PR_AUC_Rank",
            "ROC_AUC",
            "ROC_AUC_Rank",
            "Brier_Score"
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# AUTOMATICALLY IDENTIFY THE BEST MODEL
# ============================================================

best_pr_auc_model = (
    final_model_comparison
    .loc[
        final_model_comparison["PR_AUC"].idxmax(),
        "Model"
    ]
)

best_roc_auc_model = (
    final_model_comparison
    .loc[
        final_model_comparison["ROC_AUC"].idxmax(),
        "Model"
    ]
)

best_brier_model = (
    final_model_comparison
    .loc[
        final_model_comparison["Brier_Score"].idxmin(),
        "Model"
    ]
)

print("FINAL MODEL SELECTION")
print("=" * 70)

print(f"Best PR-AUC model  : {best_pr_auc_model}")
print(f"Best ROC-AUC model : {best_roc_auc_model}")
print(f"Best Brier model   : {best_brier_model}")

In [ ]:
# ============================================================
# EXPORT FINAL MODELLING RESULTS
# ============================================================

final_model_comparison.to_csv(LOCAL_OUTPUT_DIR / "final_model_comparison.csv",
    index=False
)

gb_feature_importance.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_feature_importance.csv",
    index=False
)

threshold_plot_df.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_threshold_analysis.csv",
    index=False
)

error_type_summary.to_csv(LOCAL_OUTPUT_DIR / "gradient_boosting_error_analysis.csv"
)

print("Final modelling results exported successfully.")